# Database Lens — OLTP and OLAP

A read-only window onto both databases.

* **OLTP** is RDS PostgreSQL, schema `aimternet_oltp` — current operational state: who is
  checked in, what stock is left, the points ledger.
* **OLAP** is Redshift, schema `aimternet_olap` — the analytical warehouse: facts and
  dimensions built from the Gold layer.

Safety is structural, not a promise. This notebook connects through the `aimternet_ro` role,
which holds `SELECT` and nothing else, inside a transaction pinned `READ ONLY`. An `UPDATE`
typed into a cell below fails twice over.

> Both instances are shared with unrelated coursework. Every query here is scoped to our own
> schemas.

In [ ]:
import pandas as pd
from aimternet.config.settings import settings
from aimternet.observability import db_lens

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

cfg = settings()
print(f"OLTP : {cfg.pg_host}  schema={cfg.pg_schema}  as={cfg.pg_ro_user} (read-only)")
print(f"OLAP : {cfg.redshift_host}  schema={cfg.redshift_schema}")

## 1. What is in the operational database?

In [ ]:
db_lens.oltp_tables()

In [ ]:
# Exact counts for the business tables. Compare these against the reconciliation report.
db_lens.oltp_exact_counts()

## 2. The floor right now

Open rentals are rows with no `session_end_utc`. The partial unique indexes
`one_open_rental_per_workstation` and `one_open_rental_per_member` make it impossible for a
workstation or a member to appear here twice.

In [ ]:
open_now = db_lens.open_rentals()
print(f"{len(open_now)} open rental(s)")
open_now

In [ ]:
db_lens.workstation_status()

## 3. Where the money came from

In [ ]:
revenue = db_lens.revenue_by_zone(limit_days=14)
revenue

In [ ]:
# Same data as a chart, when the table gets long enough to stop being readable.
if not revenue.empty:
    pivot = revenue.pivot_table(
        index="day", columns="zone_classification", values="net_revenue", aggfunc="sum"
    ).fillna(0)
    ax = pivot.plot(kind="bar", stacked=True, figsize=(11, 4))
    ax.set_ylabel("net revenue (PHP)")
    ax.set_title("Rental revenue by zone")
else:
    print("No rentals loaded yet — run `make bootstrap` first.")

## 4. Data-quality spot checks

Two checks worth running by hand whenever something looks off.

**The D2 cohort.** 840 members (`M-1001`–`M-1840`) are referenced by transactions but appear
in no source file. Under the `synthesize_stub` policy they are inserted as flagged stubs, so
they should be visible here rather than hidden.

In [ ]:
db_lens.backfilled_members()

**Points balance drift.** The ledger is the audit trail; `members.current_points_balance` is a
cache of it. Any row returned here means the cache disagrees with the ledger — the ledger wins.
An empty result is the healthy answer.

In [ ]:
drift = db_lens.points_balance_drift()
print("clean — every balance matches its ledger" if drift.empty else f"{len(drift)} member(s) drifting")
drift

In [ ]:
db_lens.top_members(10)

## 5. The analytical warehouse

In [ ]:
db_lens.olap_tables()

In [ ]:
# Anything in aimternet_olap is queryable from here. Once Gold is loaded, try:
#   db_lens.olap("SELECT * FROM agg_workstation_utilization_hourly LIMIT 20")
tables = db_lens.olap_tables()
if tables.empty:
    print("Redshift schema is empty — run `make redshift` after `make curate`.")
else:
    display(db_lens.olap(f"SELECT * FROM {tables.iloc[0]['table_name']} LIMIT 10"))

## 6. Your own query

`db_lens.oltp(sql)` and `db_lens.olap(sql)` both return a DataFrame. The OLTP one is
read-only; the OLAP one runs as the cluster user, so keep it to `SELECT`.

In [ ]:
db_lens.oltp('''
    SELECT current_tier, count(*) AS members, sum(lifetime_spend_amount) AS lifetime_spend
    FROM members
    GROUP BY current_tier
    ORDER BY lifetime_spend DESC NULLS LAST
''')